In [ ]:
!pip install mistralai==0.4.2 --quiet
!pip install ragas --quiet
!pip install datasets --quiet
!pip install nest_asyncio --quiet
!pip install langchain-community --quiet
!pip install langchain-mistralai --quiet

In [ ]:
# @title Imports
import os
import pandas as pd
from datasets import Dataset
import asyncio
import nest_asyncio
from typing import List, Optional, Any, Dict # Keep for type hints if needed
from langchain_mistralai.chat_models import ChatMistralAI
from langchain_mistralai.embeddings import MistralAIEmbeddings
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

# Apply nest_asyncio
nest_asyncio.apply()

In [ ]:
test_questions = [
    "What is the Mayor's name?",
    "What were the main projects in 2023?",
    "What are the City Hall opening hours?",
    "Summarize the city regulations.",
    "How many kilometers of bike lanes will we have?",
    "Hello, what's the weather like today?"
]
answers = [
    "The Mayor's name is Mrs. Sparkle Merriweather.",
    "Central road renovation: Overhaul and rehabilitation of the main downtown streets (started 2023-09-01, budget of $750,000). Public lighting modernization: Replacement of faulty street lamps and installation of LEDs (started 2023-10-15, budget of $300,000). Green spaces improvement: Redevelopment of municipal parks and gardens (started 2023-11-10, budget of $450,000). Elementary school renovation: Modernization and extension of the town's elementary school facilities (started 2023-08-20, budget of $650,000). Creation of the Municipal Sports Center: Construction of a multipurpose sports complex (started 2023-07-15, budget of $950,000). Market Square redevelopment: Reconfiguration of the public space to boost local commerce (started 2023-10-05, budget of $500,000). Installation of an urban surveillance system: Deployment of cameras to strengthen public safety (started 2023-12-01, budget of $400,000). Development of bike lanes: Creation of a secure network of bike lanes throughout the town (started 2023-09-15, budget of $350,000). Setting up an advanced selective sorting system: Optimization of waste management through a smart sorting system (started 2023-11-20, budget of $300,000).",
    "Willow Creek City Hall is open Monday to Friday from 8:30 AM to 12:00 PM. It is closed on Saturday and Sunday.",
    "The regulations aim to ensure public order, safety, peace and sanitation across the whole town. They apply to all residents, visitors and users of the town.",
    "The 2026 action plan aims to create approximately 2.5 km of new secure bike lanes.",
    "Hello! I cannot provide real-time weather information. To find out the current weather in Willow Creek, I recommend checking a reliable weather website or app."
]

placeholder_contexts = [
    ["Excerpt from the municipal document page 5: The City Council elected Mrs. Sparkle Merriweather as Mayor during the June 15 session."],
    ["2023 activity report, Works section: Central road renovation ($750k), LED public lighting ($300k), Parks redevelopment ($450k).", "2023 budget appendix: Elementary school ($650k), Sports Center ($950k), Market Square ($500k), Surveillance cameras ($400k), Bike lanes ($350k), Selective sorting ($300k)."],
    ["'Practical Info' page of the website: City Hall hours: Monday to Friday: 8:30 AM - 12:00 PM. Saturday, Sunday: Closed."],
    ["Article 1 of the City Regulations: These regulations aim to ensure good order, safety, security, peace and public sanitation within the town.", "Article 2: They apply to any person located within the town."],
    ["Soft Mobility Plan 2022-2026, page 12: 2026 target: +2.5 km of secure bike lanes (paths and lanes)."],
    ["Internal documentation of the conversational agent: Known limits: Do not provide real-time information such as weather, stock market, or road traffic. Suggest external sources."]
]
ground_truths = [
    "The Mayor's name is Mrs. Sparkle Merriweather.",
    "The main projects in 2023 concern roads, public lighting, green spaces, the elementary school, a sports center, the market square, urban surveillance, bike lanes and selective sorting.",
    "City Hall is open Monday to Friday from 8:30 AM to 12:00 PM.",
    "The city regulations mainly aim to ensure public order, safety, peace and sanitation in the town for everyone.",
    "The 2026 action plan provides for approximately 2.5 km of new secure bike lanes.",
    "The system cannot give the weather in real time and suggests checking an external source."
]
evaluation_data = {
    "question": test_questions,
    "answer": answers,
    "contexts": placeholder_contexts,
    "ground_truth": ground_truths # Make sure this key matches what Ragas expects (sometimes 'reference' or other)
}


# Convert to a Hugging Face Dataset
evaluation_dataset = Dataset.from_dict(evaluation_data)
print("Dataset created.")

In [ ]:
# @title Running the Ragas Evaluation

MISTRAL_API_KEY = "StthE16bt5IReMHPxgjbwtQLGggjOGtq"


try:
    # 1. Instantiate the LLM and the Embeddings via Langchain (SIMPLE)
    print("Initializing the ChatMistralAI LLM...")
    mistral_llm = ChatMistralAI(
        mistral_api_key=MISTRAL_API_KEY,
        model="mistral-large-latest", # Or another supported chat model
        temperature=0.1
    )
    print("LLM initialized.")

    print("\nInitializing the MistralAIEmbeddings...")
    mistral_embeddings = MistralAIEmbeddings(
        mistral_api_key=MISTRAL_API_KEY
        # model="mistral-embed" # The default model is generally correct
    )
    print("Embeddings initialized.")

    # 2. Define the metrics (the names are generally the same)
    metrics_to_evaluate = [
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ]
    print(f"\nSelected metrics: {[m.name for m in metrics_to_evaluate]}")

    # 3. Launch the Ragas evaluation

    print("\nLaunching the Ragas evaluation ...")
    results = evaluate(
        dataset=evaluation_dataset,
        metrics=metrics_to_evaluate,
        llm=mistral_llm,              # Standard Langchain instance
        embeddings=mistral_embeddings # Standard Langchain instance
        # Ragas handles the async/sync calls internally with the Langchain objects
    )
    print("\n--- Ragas evaluation finished ---")

    # 4. Display the results
    print("\n--- Results ---")
    print(results) # The output format may vary slightly across versions

    # Optional: DataFrame display if the structure allows it
    if isinstance(results, dict) and 'ragas_score' in results:
            print(f"\nGlobal RAGAS score: {results['ragas_score']:.4f}")
            # Display the per-metric scores if they are directly available
            for metric in metrics_to_evaluate:
                if metric.name in results:
                    print(f"Score {metric.name}: {results[metric.name]:.4f}")
    else:
        try:
                # Try to convert to a DataFrame if it is a Dataset or similar
                results_df = results.to_pandas()
                print("\n--- Results (DataFrame) ---")
                pd.set_option('display.max_columns', None)
                print(results_df.head(len(results_df)))
        except AttributeError:
                print("\nResult format not directly convertible to a Pandas DataFrame.")


except Exception as e:
    print(f"\nAn error occurred during the initialization or the evaluation: {e}")
    import traceback
    print("\nTraceback:")
    traceback.print_exc()